In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/subjects_AND_sampling_metadata.csv", parse_dates=["created_at"])

C:\Users\yuyue\AppData\Local\Temp\ipykernel_25252\87986443.py:1: DtypeWarning: Columns (0: description, 1: entities.url.urls, 2: entities.description.mentions, 3: entities.description.hashtags, 4: entities.description.urls, 5: url, 6: entities.urls.subject_pool, 7: entities.mentions.subject_pool, 8: entities.annotations.subject_pool, 9: entities.hashtags.subject_pool, 10: edit_history_tweet_ids.subject_pool, 11: attachments.media_keys.subject_pool, 12: referenced_tweets.subject_pool, 13: entities.cashtags.subject_pool, 14: entities.description.cashtags, 15: geo.place_id.subject_pool, 16: entities.hashtags.tweets_historical, 17: edit_history_tweet_ids.tweets_historical, 18: attachments.media_keys.tweets_historical, 19: attachments.media_source_tweet_id.tweets_historical, 20: geo.place_id.tweets_historical, 21: entities.cashtags.tweets_historical, 22: article.title, 23: geo.coordinates.type, 24: geo.coordinates.coordinates) have mixed types. Specify dtype option on import or set low_memo

In [4]:
# filter the dataframe to only include tweets from the year leading up to the election date
election_date = pd.Timestamp("2024-11-05", tz="UTC")
start_period = election_date - pd.DateOffset(years=1)
end_period = election_date
df = df[(df["created_at"] >= start_period) & (df["created_at"] <= end_period)].copy()

In [5]:
WINDOW_DAYS = 7  # variable to adjust time window size

df['created_at'] = pd.to_datetime(df['created_at'], utc=True)

# Create WINDOW_DAYS-day windows starting from first available date
first_date = df['created_at'].min()
df['days_since_start'] = (df['created_at'] - first_date).dt.days
df['window_number'] = (df['days_since_start'] // WINDOW_DAYS).astype(int)
df['time_window_start'] = first_date + pd.to_timedelta(df['window_number'] * WINDOW_DAYS, unit='D')
df['window_weeks_to_election'] = ((election_date - df['time_window_start']).dt.days // 7).astype(int)

# Build per-tweet dictionary
df['tweet_dict'] = df.apply(
    lambda row: {
        'text': row['tweets_historical'],
        'created_at': row['created_at'].tz_convert(None)
    },
    axis=1
)

# Group tweets by user and window
grouped = df.groupby(['id', 'time_window_start'])

def combine_tweets(group):
    weeks_to_election = int(group['window_weeks_to_election'].iloc[0])
    tweets_list = group['tweet_dict'].tolist()
    tweets_list.insert(0, f"{weeks_to_election} weeks to election")
    return tweets_list

# Aggregate tweets per window
tweets_df = grouped.apply(combine_tweets).reset_index(name='tweets_struct')

# Get constant user metadata
user_meta_cols = ['id', 'name', 'username', 'description', 'location',
                  'public_metrics.followers_count', 'profile_image_url', 'clean...pool']
user_meta_df = df[user_meta_cols].drop_duplicates(subset=['id'])

# Fill missing windows for all users
all_users = df['id'].unique()
all_windows = pd.date_range(first_date,
                            df['time_window_start'].max(),
                            freq=f'{WINDOW_DAYS}D')
full_index = pd.MultiIndex.from_product([all_users, all_windows],
                                        names=['id', 'time_window_start'])
full_df = pd.DataFrame(index=full_index).reset_index()

# Merge tweets per window
full_df = full_df.merge(tweets_df, on=['id', 'time_window_start'], how='left')
full_df = full_df.merge(user_meta_df, on='id', how='left')

# Fill missing tweets_struct with empty list
full_df['tweets_struct'] = full_df['tweets_struct'].apply(lambda x: x if isinstance(x, list) else [])
full_df['is_empty_window'] = full_df['tweets_struct'].apply(lambda x: len(x) == 0)

# Compute weeks to election for all windows
full_df['window_weeks_to_election'] = ((election_date - full_df['time_window_start']).dt.days // 7).astype(int)

In [6]:
full_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,is_empty_window,window_weeks_to_election
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,52
1,1081532502966239232,2023-11-12 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,51
2,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,50
3,1081532502966239232,2023-11-26 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,49
4,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,48
...,...,...,...,...,...,...,...,...,...,...,...,...
48251,142827500,2024-09-29 00:00:00+00:00,"[5 weeks to election, {'text': '@kevin__fm Sis...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,5
48252,142827500,2024-10-06 00:00:00+00:00,"[4 weeks to election, {'text': '@jacuzzicoaste...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,4
48253,142827500,2024-10-13 00:00:00+00:00,"[3 weeks to election, {'text': '@cherryrepress...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,3
48254,142827500,2024-10-20 00:00:00+00:00,"[2 weeks to election, {'text': '@bong_rot beau...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,2


In [9]:
df_with_desc = pd.read_csv("../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/final_user_profiles_cleaned.csv")  

# Merge on "id"
merged_df = full_df.merge(
    df_with_desc[["id", "profile_image_url", "profile_image_description", "profile_image_main_info"]],
    on="id",
    how="left",
    suffixes=("", "_new")  
)

# Overwrite profile_image_url with the new value 
merged_df["profile_image_url"] = merged_df["profile_image_url_new"].combine_first(merged_df["profile_image_url"])

# Drop the temporary column used for overwrite
merged_df = merged_df.drop(columns=["profile_image_url_new"])

merged_df.to_csv(f"../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/all_user_windows_{WINDOW_DAYS}d.csv", index=False)
merged_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,is_empty_window,window_weeks_to_election,profile_image_description,profile_image_main_info
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,52,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
1,1081532502966239232,2023-11-12 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,51,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
2,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,50,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
3,1081532502966239232,2023-11-26 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,49,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
4,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,48,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48251,142827500,2024-09-29 00:00:00+00:00,"[5 weeks to election, {'text': '@kevin__fm Sis...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,5,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
48252,142827500,2024-10-06 00:00:00+00:00,"[4 weeks to election, {'text': '@jacuzzicoaste...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,4,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
48253,142827500,2024-10-13 00:00:00+00:00,"[3 weeks to election, {'text': '@cherryrepress...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,3,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
48254,142827500,2024-10-20 00:00:00+00:00,"[2 weeks to election, {'text': '@bong_rot beau...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,2,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...


In [10]:
# First, check what type the data actually is
print(f"Type of tweets_struct: {type(full_df['tweets_struct'].iloc[100])}")

# If it's a string, check its length and first few characters
if isinstance(full_df['tweets_struct'].iloc[100], str):
    print(f"String length: {len(full_df['tweets_struct'].iloc[100])}")
    print(f"First 200 characters: {full_df['tweets_struct'].iloc[100]}")

Type of tweets_struct: <class 'list'>


In [11]:
import matplotlib.pyplot as plt

num_users = merged_df["id"].nunique()
print(f"Number of unique users sampled: {num_users}")

num_windows = merged_df.groupby(["id", "time_window_start"]).ngroups
print(f"Number of sampled windows: {num_windows}")

Number of unique users sampled: 928
Number of sampled windows: 48256


In [12]:
# Number of unique windows per user
windows_per_user = full_df.groupby('id')['time_window_start'].nunique()

# Compute stats
avg_windows = windows_per_user.mean()
min_windows = windows_per_user.min()
max_windows = windows_per_user.max()
std_windows = windows_per_user.std()

print(f"Average windows per user: {avg_windows:.2f}")
print(f"Min windows per user: {min_windows}")
print(f"Max windows per user: {max_windows}")
print(f"Standard deviation: {std_windows:.2f}")

Average windows per user: 52.00
Min windows per user: 52
Max windows per user: 52
Standard deviation: 0.00


In [13]:
empty_percentage = (full_df['is_empty_window'].mean() * 100).round(2)
print(f"Percentage of empty windows: {empty_percentage}%")

Percentage of empty windows: 59.01%


In [14]:
print(full_df.info())   
print(full_df.head())  
print(full_df.tail())

<class 'pandas.DataFrame'>
RangeIndex: 48256 entries, 0 to 48255
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype              
---  ------                          --------------  -----              
 0   id                              48256 non-null  int64              
 1   time_window_start               48256 non-null  datetime64[us, UTC]
 2   tweets_struct                   48256 non-null  object             
 3   name                            48256 non-null  str                
 4   username                        48256 non-null  str                
 5   description                     45968 non-null  str                
 6   location                        48256 non-null  str                
 7   public_metrics.followers_count  48256 non-null  int64              
 8   profile_image_url               48256 non-null  str                
 9   clean...pool                    48256 non-null  str                
 10  is_empty_window      